### Schema Evolution
##### 1.Adding New Columns (Manual/Automatic)
##### 2.Widening Data types
###### a.Int to Bigint
###### b.float to double
###### c.varchar(10) to varchar(20)
##### 3.Nested Structure Evolution
##### 4.Column position changes

#### 1.Adding new columns

In [0]:
drop table if exists deltacatalog.deltadb.invoices_se;
create table if not exists deltacatalog.deltadb.invoices_se
(
  customer_id int not null,
  invoice_no string,
  price float,
  invoice_date date
)

In [0]:
insert into deltacatalog.deltadb.invoices_se
select customer_id,invoice_no,price,invoice_date from parquet.`abfss://labdata@stgdatabricksdelta.dfs.core.windows.net/invoices/invoices_1_100.parquet`
where customer_id <= 5

In [0]:
alter table deltacatalog.deltadb.invoices_se
add columns quantity int;

In [0]:
insert into deltacatalog.deltadb.invoices_se
select customer_id,invoice_no,price,invoice_date,quantity from parquet.`abfss://labdata@stgdatabricksdelta.dfs.core.windows.net/invoices/invoices_1_100.parquet`
where customer_id between 6 and 10

In [0]:
set spark.databricks.delta.schema.autoMerge.enabled = true

In [0]:
insert into deltacatalog.deltadb.invoices_se
select customer_id,invoice_no,price,invoice_date,quantity,payment_method from parquet.`abfss://labdata@stgdatabricksdelta.dfs.core.windows.net/invoices/invoices_1_100.parquet`
where customer_id between 11 and 15

#### 2.Data Type Widening

In [0]:
alter table deltacatalog.deltadb.invoices_se
set tblproperties ("delta.enableTypeWidening" = "true")

In [0]:
DESCRIBE TABLE deltacatalog.deltadb.invoices_se;

In [0]:
insert into deltacatalog.deltadb.invoices_se
values (123456789012345123,'I89735',162.63999938964844,'2025-09-22',200,"Cash")

In [0]:
alter table deltacatalog.deltadb.invoices_se
alter column customer_id type bigint

In [0]:
insert into deltacatalog.deltadb.invoices_se
values (123456789012345123,'I89735',162.63999938964844,'2025-09-22',200,"Cash")

#### 3. Nested Structure Evolution

In [0]:
alter table deltacatalog.deltadb.invoices_se
add columns payment_details struct<pincode int,storecode int>

In [0]:
insert into deltacatalog.deltadb.invoices_se
values (16,"I765438",48.90,'2025-09-23',20,"Credit Card",struct(1345,150))

In [0]:
Alter table deltacatalog.deltadb.invoices_se
alter column payment_details.pincode type bigint;

In [0]:
insert into deltacatalog.deltadb.invoices_se
values (17,"I984356",57.90,'2025-09-23',20,"Cash",struct(123456789012345123,150))

In [0]:
Alter table deltacatalog.deltadb.invoices_se
add column payment_details.storelocation string;

In [0]:
insert into deltacatalog.deltadb.invoices_se
values (18,"I984356",57.90,'2025-09-23',20,"Cash",struct(123456789012345123,150,"San Dieago"))

In [0]:
insert into deltacatalog.deltadb.invoices_se
values (19,"I76453",84.5,'2025-09-23',20,"Credit Card",
named_struct(
  "pincode",123456789012345123,
  "storecode",150,
  "storelocation","San Dieago",
  "staffid","ST1234"))

#### Column Position Changes

In [0]:
set spark.databricks.delta.schema.autoMerge.enabled = false

In [0]:
Alter table deltacatalog.deltadb.invoices_se
add columns (age int after price);

In [0]:
insert into deltacatalog.deltadb.invoices_se
select customer_id,invoice_no,price,age,invoice_date,quantity,payment_method,struct(100,200,"India","Stf123")
from parquet.`abfss://labdata@stgdatabricksdelta.dfs.core.windows.net/invoices/invoices_1_100.parquet`
where customer_id between 40 and 45

In [0]:
insert into deltacatalog.deltadb.invoices_se
select customer_id,invoice_no,price,age,invoice_date,quantity,payment_method,gender,null,category
from parquet.`abfss://labdata@stgdatabricksdelta.dfs.core.windows.net/invoices/invoices_1_100.parquet`
where customer_id between 55 and 60


In [0]:
merge into deltacatalog.deltadb.invoices_se tgt
using (select customer_id,invoice_no,price,age,invoice_date,quantity,payment_method,gender,null as payment_details,category
from parquet.`abfss://labdata@stgdatabricksdelta.dfs.core.windows.net/invoices/invoices_1_100.parquet`
where customer_id between 55 and 60) src
on tgt.customer_id = src.customer_id
when not matched then insert *

In [0]:
%python
from pyspark.sql.functions import col
df = spark.read.parquet("abfss://labdata@stgdatabricksdelta.dfs.core.windows.net/invoices/invoices_1_100.parquet") \
          .where(col("customer_id").between(1, 10)) \
          .select(col("customer_id"),col("invoice_no"),col("invoice_date"))
           
df.write.saveAsTable("deltacatalog.deltadb.invoices_spark")

In [0]:
%python
from pyspark.sql.functions import col
df = spark.read.parquet("abfss://labdata@stgdatabricksdelta.dfs.core.windows.net/invoices/invoices_1_100.parquet") \
          .where(col("customer_id").between(11, 20)) \
          .select(col("customer_id"),col("invoice_no"),col("invoice_date"),col("quantity"),col("price"))
           
df.write.mode("append").option("mergeSchema","true").saveAsTable("deltacatalog.deltadb.invoices_spark")

In [0]:
select * from deltacatalog.deltadb.invoices_spark;

In [0]:
select * from deltacatalog.deltadb.invoices_se